In [1]:
###Cell 1: install dependencies, including a newer release of the MLflow SDK

! pip install --upgrade -i https://pypi.org/simple openai mlflow[kubernetes]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 301.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 263.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 534.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 615.4 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.41.1
    Uninstalling openai-2.41.1:
      Successfully uninstalled openai-2.41.1
  Attempting uninstall: mlflow-tracing━━━━━━━━━━ 0/4 [openai]
    Found existing installation: mlflow-tracing 3.13.00/4 [openai]
    Uninstalling mlflow-tracing-3.13.0:━━━━━ 0/4 [openai]
      Successfully uninstalled mlflow-tracing-3.13.0━━━━━━━━━━━━━━ 1/4 [mlflow-tracing]
  Attempting uninstall: mlflow-skinny━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [mlflow-tracing]
    Found existing installation: mlflow-skinny 3.13.0━━━━━━━━━ 1/4 [mlflow-tracing]
    Uninstalling mlflow-skinny-3.13.0:━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [mlflow-tracing]
      Succes

In [2]:
###Cell 2: hardcoded prompts

input_text = """
What's the difference between RHEL and CentOS?
"""
system_instructions = """
You are a helpful AI assistant.
You are designed to answer questions in a concise and professional manner.
"""

better_instructions = """
Answer questions with short answers, of one to three short sentences.
"""

In [3]:
###Cell 3: environment settings

import os

BASE_URL = "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1"
model_name = "llama-32-3b-instruct"

MLFLOW_TRACKING_URL = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
MLFLOW_EXPERIMENT = "simple-agent"

os.environ["MLFLOW_TRACKING_AUTH"]="kubernetes-namespaced"

JUDGE_URL = BASE_URL
judge_model = model_name
os.environ["OPENAI_API_BASE"] = JUDGE_URL
os.environ["OPENAI_API_KEY"] = "no token"

In [4]:
###Cell 4: start tracing

import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URL)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tracing.disable_notebook_display()
mlflow.openai.autolog()

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
###Cell 5: simplistic agent

from openai import OpenAI

def agent (key: str, system: str, query: str) -> str:
    client = OpenAI(
        base_url=BASE_URL,
        api_key=key
    )
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": query}
        ],
        max_tokens=256,
        temperature=0.7
    )

    return response.choices[0].message.content

In [6]:
###Cell 6: run the agent

# don't need this because the inferenceservice is setup with token auth disabled
#try:
#    with open("/var/run/secrets/kubernetes.io/serviceaccount/token", "r") as f:
#        AUTH_TOKEN = f.read().strip()
#except FileNotFoundError:
#    # Fallback if running outside a workbench or using a manually generated cluster token
#    AUTH_TOKEN = OCP_TOKEN

try:
    system_prompt = mlflow.genai.load_prompt("prompts:/short-responses/1")
    formatted_prompt = system_prompt.format()
    response = agent("no token", formatted_prompt, input_text)    
    print(response)
except Exception as e:
    print(f"An error occurred: {e}")

RHEL (Red Hat Enterprise Linux) and CentOS are both Linux distributions, but they have distinct differences. RHEL is a proprietary distribution developed by Red Hat, while CentOS is a community-driven, open-source distribution based on RHEL. CentOS is no longer actively supported after 2020, but it remains a popular alternative to RHEL.


In [7]:
###Cell 7: LLM judges

from mlflow.genai.scorers import Correctness
from mlflow.genai.judges import make_judge

correctness_judge = Correctness(
    model=f"openai:/{judge_model}"
)

acronyms_judge = make_judge(
    name="expanded_acronyms",
    instructions=(
        "Check if this text expands any common IT acronyms or product names:\n"
        "{{ outputs }}"
        "\n\nEVALUATION STEPS:\n"
        "1. Scan the text for IT acronyms (e.g., TCP, IP, AI, LLM) or similar names (e.g., Vim, GNOME, LAME, RHEL).\n"
        "2. Check if the text explicitly expands them to their full words (e.g., 'LLM (Large-Language Model)', 'GNOME (GNU Network Object Model Environment).\n"
        "3. Merely explaining or describing a concept (e.g., 'Vim is a text editor') (e.g. 'RHEL is an operating system') does NOT count as an expansion.\n\n"
        "BOOLEAN OUTPUT:\n"
        "- Return True if you found ONE OR MORE acronym expansions.\n"
        "- Return False if the response is completely FREE of acronym expansions (NO expansions found).\n"
    ),
    feedback_value_type=bool,
    model=f"openai:/{judge_model}",
    inference_params={"temperature": 0.0},
    base_url=f"{JUDGE_URL}/chat/completions"
)

In [8]:
###Cell 8: test judges

print ("Test built-in judge:\n")
feedback = correctness_judge(
    inputs={"question": "What is RHEL?"},
    outputs={"response": "RHEL is a Linux distribution from Red Hat."},
    expectations={"expected_response": "RHEL (Red Hat Enterprise Linux) is a Linux distribution."},
)
print(f"• Test judge with correct definition of RHEL: {feedback.value}\n  {feedback.rationale}")

feedback = correctness_judge(
    inputs={"question": "What is RHEL?"},
    outputs={"response": "RHEL (Rebooting Helps Every Lag) is pun on IT struggles."},
    expectations={"expected_response": "RHEL (Red Hat Enterprise Linux) is a Linux distribution."},
)
print(f"• Test judge with incorrect definition of RHEL: {feedback.value}\n  {feedback.rationale}")

print ("\nTest LLM judge:\n")

feedback = acronyms_judge(
    inputs={"question": "What is RHEL?"},
    outputs={"response": "RHEL is a Linux distribution."},
)
print(f"• Test judge with RHEL (no expansion): {feedback.value}\n  {feedback.rationale}")

feedback = acronyms_judge(
    inputs={"question": "Who supports RHEL?"},
    outputs={"response": "Red Hat provides commercial support for users of Linux."},
)
print(f"• Test judge with RHEL (no expansion): {feedback.value}\n  {feedback.rationale}")

feedback = acronyms_judge(
    inputs={"question": "What is RHEL?"},
    outputs={"response": "RHEL (Red Hat Enterprise Linux) is a Linux distribution."},
)
print(f"• Test judge with RHEL (expanded)': {feedback.value}\n  {feedback.rationale}")

Test built-in judge:

• Test judge with correct definition of RHEL: yes
  The first statement in the claim is 'RHEL' which is supported by the document as the question and response both mention 'RHEL'. The second statement in the claim is 'is a Linux distribution' which is also supported by the document as the response explicitly states 'a Linux distribution from Red Hat'.
• Test judge with incorrect definition of RHEL: no
  The claim states RHEL is a Linux distribution. The document does not mention Linux distribution, it mentions a pun on IT struggles.

Test LLM judge:

• Test judge with RHEL (no expansion): False
  The text does not explicitly expand any IT acronyms or product names. Although it mentions 'RHEL', it only provides a brief description without expanding the acronym to its full word 'Linux distribution'. The text does not meet the criteria for an acronym expansion.
• Test judge with RHEL (no expansion): False
  The text does not explicitly expand any IT acronyms or produ

In [9]:
###Cell 9: hardcoded evaluation data set

eval_dataset = [
    {
        "inputs": {"question":
            "What's the difference between RHEL and CentOS?"},
        "expectations": {"expected_response":
            """
            RHEL and CentOS are both Linux distributions.
            RHEL is a commercial distribution from Red Hat, while CentOS is a free distribution.
            CentOS Stream is a direct upstream of RHEL.
            """},
    },
    {
        "inputs": {"question":
            "What's the difference between RHEL and Fedora Linux?"},
        "expectations": {"expected_response":
            """
            RHEL and Fedora are both Linux distributions.
            RHEL is a commercial distribution from Red Hat, while Fedora is a free distribution.
            Fedora is an upstream of RHEL.
            """},
    },
]


In [10]:
###Cell 10: run evaluation

def my_predict_fn(question: str) -> str:
    system_prompt = mlflow.genai.load_prompt("prompts:/short-responses/1")
    final_prompt = system_prompt.format()
    return agent("no token", final_prompt, input_text)

os.environ["MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION"]="true"

try:
    results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=my_predict_fn,
        scorers=[correctness_judge,acronyms_judge],
    )
except Exception as e:
    print(f"An error occurred: {e}")

2026/08/03 16:47:51 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:08, Remaining: 00:00] [predict_fn: 22%, scorers: 78%]
